In [ ]:
import glob
import pandas as pd
import sqlite3

In [ ]:
conn = sqlite3.connect(":memory:")
files = glob.glob("player_stats/*.csv")

mvp_votes = {
    "2005": {
        "Shaun Alexander": 0.85,
        "Peyton Manning": 0.26,
        "Tom Brady": 0.20,
        "Tiki Barber": 0.12,
        "Carson Palmer": 0.04
    },
    "2006": {
        "LaDainian Tomlinson": 0.88,
        "Drew Brees": 0.08,
        "Peyton Manning": 0.04
    },
    "2007": {
        "Tom Brady": 0.98,
        "Brett Favre": 0.02
    },
    "2008": {
        "Peyton Manning": 0.64,
        "Chad Pennington": 0.08,
        "Michael Turner": 0.08,
        "James Harrison": 0.06,
        "Adrian Peterson": 0.06,
        "Philip Rivers": 0.04,
        "Chris Johnson": 0.02,
        "Kurt Warner": 0.02
    },
    "2009": {
        "Peyton Manning": 0.79,
        "Drew Brees": 0.15,
        "Philip Rivers": 0.04,
        "Brett Favre": 0.02
    },
    "2010": {
        "Tom Brady": 1
    },
    "2011": {
        "Aaron Rodgers": 0.96,
        "Drew Brees": 0.04
    },
    "2012": {
        "Adrian Peterson": 0.61,
        "Peyton Manning": 0.39
    },
    "2013": {
        "Peyton Manning": 0.98,
        "Tom Brady": 0.02
    },
    "2014": {
        "Aaron Rodgers": 0.62,
        "J.J. Watt": 0.26,
        "DeMarco Murray": 0.04,
        "Tony Romo": 0.04,
        "Tom Brady": 0.02,
        "Bobby Wagner": 0.02
    },
    "2015": {
        "Cam Newton": 0.96,
        "Tom Brady": 0.02,
        "Carson Palmer": 0.02
    },
    "2016": {
        "Matt Ryan": 0.50,
        "Tom Brady": 0.20,
        "Derek Carr": 0.12,
        "Ezekiel Elliott": 0.12,
        "Aaron Rodgers": 0.04,
        "Dak Prescott": 0.02
    },
    "2017": {
        "Tom Brady": 0.80,
        "Todd Gurley": 0.16,
        "Carson Wentz": 0.04
    },
    "2018": {
        "Patrick Mahomes": 0.82,
        "Drew Brees": 0.18
    },
    "2019": {
        "Lamar Jackson": 1.00
    },
    "2020": {
        "Aaron Rodgers": 0.88,
        "Josh Allen": 0.08,
        "Patrick Mahomes": 0.04
    },
    "2021": {
        "Aaron Rodgers": 0.78,
        "Tom Brady": 0.10,
        "Cooper Kupp": 0.02
    }
}


for file in files:
    df = pd.read_csv(file)
    year = file.split("_")[-1].split(".")[0]
    df["mvp_vote_share"] = df["player_display_name"].map(mvp_votes[year])
    df["mvp_vote_share"] = df["mvp_vote_share"].fillna(0)
    df.to_sql(f"stats_{year}", conn, index=False, if_exists="replace")

years = sorted([file.split("_")[-1].split(".")[0] for file in files])

cols_to_drop = ["special_teams_tds", "headshot_url", "position_group"]

# get columns from one table (they should match across years)
first_year = years[0]
cols = pd.read_sql(f"PRAGMA table_info(stats_{first_year});", conn)
cols = [c[1] for c in cols.values]

keep_cols = [c for c in cols if c not in cols_to_drop]

query = " UNION ALL ".join([
    f"SELECT {', '.join(keep_cols)} FROM stats_{year}"
    for year in years
])

final_query = f"""
CREATE TABLE all_players_cleaned AS
{query}
"""

conn.execute(final_query)